<a href="https://colab.research.google.com/github/ebitnet65/Exciton-Code/blob/master/MonteCarloIsing_slices_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List
from datetime import datetime
import time

import pandas as pd
import seaborn as sns

sns.set_theme()

@dataclass
class IsingResults:
    beta: float
    h: float
    m_mean: float
    e_mean: float
    m_abs_mean: float
    chi: float
    c_h: float
    omega_beta_h: float
    m_err: float
    chi_err: float
    c_h_err: float
    omega_err: float


def initialize_lattice(L: int, rng: np.random.Generator) -> np.ndarray:
    """Random +/-1 initial lattice."""
    return rng.choice(np.array([-1, 1], dtype=np.int8), size=(L, L))


def total_energy(spins: np.ndarray, J: float, h: float) -> float:
    """
    Total energy:
        E = -J sum_<ij> s_i s_j - h sum_i s_i
    Periodic boundaries. Bonds counted once.
    """
    nn_sum = (
        np.sum(spins * np.roll(spins, 1, axis=0))
        + np.sum(spins * np.roll(spins, 1, axis=1))
    )
    return -J * nn_sum - h * np.sum(spins)


def metropolis_sweep(
    spins: np.ndarray,
    beta: float,
    J: float,
    h: float,
    rng: np.random.Generator
) -> None:
    """
    One Metropolis sweep: N attempted single-spin flips.
    Periodic boundaries.
    """
    L = spins.shape[0]

    for _ in range(L * L):
        i = rng.integers(L)
        j = rng.integers(L)

        s = spins[i, j]
        nn = (
            spins[(i + 1) % L, j]
            + spins[(i - 1) % L, j]
            + spins[i, (j + 1) % L]
            + spins[i, (j - 1) % L]
        )

        dE = 2.0 * s * (J * nn + h)

        if dE <= 0.0 or rng.random() < np.exp(-beta * dE):
            spins[i, j] = -s


def block_stats(x: np.ndarray, n_blocks: int = 20):
    """
    Mean and standard error using simple blocking.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    block_size = n // n_blocks

    if block_size < 2:
        return np.mean(x), np.std(x, ddof=1) / np.sqrt(max(n, 1))

    trimmed = x[: block_size * n_blocks]
    blocks = trimmed.reshape(n_blocks, block_size).mean(axis=1)

    mean = blocks.mean()
    err = blocks.std(ddof=1) / np.sqrt(n_blocks)

    return mean, err


def run_ising_point(
    L: int,
    beta: float,
    h: float,
    J: float = 1.0,
    n_therm: int = 5000,
    n_sweeps: int = 20000,
    sample_every: int = 10,
    seed: int = 1234,
    init_spins: np.ndarray | None = None,
) -> tuple[IsingResults, np.ndarray]:
    """
    Run one Ising simulation point and compute:
      m = <M>/N
      e = <E>/N
      chi = beta N Var(m)
      C_h = beta^2 N Var(e)
      Omega_beta_h = -N Cov(e,m)

    Returns results and final spin configuration, so you can warm-start nearby beta.
    """
    rng = np.random.default_rng(seed)

    if init_spins is None:
        spins = initialize_lattice(L, rng)
    else:
        spins = init_spins.copy()

    N = L * L

    for _ in range(n_therm):
        metropolis_sweep(spins, beta, J, h, rng)

    m_samples = []
    e_samples = []

    for sweep in range(n_sweeps):
        metropolis_sweep(spins, beta, J, h, rng)

        if sweep % sample_every == 0:
            M = np.sum(spins)
            E = total_energy(spins, J, h)

            m_samples.append(M / N)
            e_samples.append(E / N)

    m_samples = np.asarray(m_samples)
    e_samples = np.asarray(e_samples)

    m_mean = np.mean(m_samples)
    e_mean = np.mean(e_samples)
    m_abs_mean = np.mean(np.abs(m_samples))

    var_m_series = (m_samples - m_mean) ** 2
    var_e_series = (e_samples - e_mean) ** 2
    cov_em_series = (e_samples - e_mean) * (m_samples - m_mean)

    chi_series = beta * N * var_m_series
    c_h_series = beta**2 * N * var_e_series
    omega_series = -N * cov_em_series

    _, m_err = block_stats(m_samples)
    chi, chi_err = block_stats(chi_series)
    c_h, c_h_err = block_stats(c_h_series)
    omega, omega_err = block_stats(omega_series)

    result = IsingResults(
        beta=beta,
        h=h,
        m_mean=m_mean,
        e_mean=e_mean,
        m_abs_mean=m_abs_mean,
        chi=chi,
        c_h=c_h,
        omega_beta_h=omega,
        m_err=m_err,
        chi_err=chi_err,
        c_h_err=c_h_err,
        omega_err=omega_err,
    )

    return result, spins


def scan_beta_for_field(
    L: int,
    betas: np.ndarray,
    h: float,
    J: float = 1.0,
    n_therm: int = 5000,
    n_sweeps: int = 20000,
    sample_every: int = 10,
    seed: int = 1234,
) -> List[IsingResults]:
    """
    Scan beta at fixed h. Uses warm starts from one beta to the next.
    """
    results = []
    spins = None
    t0 = time.time()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for k, beta in enumerate(betas):
        result, spins = run_ising_point(
            L=L,
            beta=float(beta),
            h=float(h),
            J=J,
            n_therm=n_therm,
            n_sweeps=n_sweeps,
            sample_every=sample_every,
            seed=seed + 1000 * k + int(1e5 * h),
            init_spins=spins,
        )

        results.append(result)
        elapsed = time.time()-t0

        print(
            f"[{now}]"
            f"h={h:.3f}, beta={beta:.4f}, "
            f"m={result.m_mean:+.4f}, chi={result.chi:.3f}, "
            f"C={result.c_h:.3f}, Omega={result.omega_beta_h:.3f}"
            f" (elapsed={elapsed/60:.1f}min)"
        )

        pd.DataFrame(
    [{
        "beta": r.beta,
        "h": r.h,
        "m": r.m_mean,
        "chi": r.chi,
        "C": r.c_h,
        "Omega": r.omega_beta_h
    }
    for r in results]
    ).to_csv(
    f"checkpoint_h_{h:.3f}.csv",
    index=False
    )

    return results


def results_to_arrays(results: List[IsingResults]) -> Dict[str, np.ndarray]:
    keys = IsingResults.__dataclass_fields__.keys()
    return {key: np.array([getattr(r, key) for r in results]) for key in keys}


def plot_cross_sections(all_data: Dict[float, Dict[str, np.ndarray]]):
    """
    Make cross-section plots for m, chi, C_h, Omega_beta_h vs beta.
    """
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)

    ax_m, ax_chi, ax_c, ax_om = axes.ravel()

    for h, data in all_data.items():
        beta = data["beta"]

        ax_m.errorbar(
            beta,
            data["m_mean"],
            yerr=data["m_err"],
            marker="o",
            lw=1.5,
            ms=4,
            capsize=2,
            label=fr"$h={h:g}$",
        )

        ax_chi.errorbar(
            beta,
            data["chi"],
            yerr=data["chi_err"],
            marker="o",
            lw=1.5,
            ms=4,
            capsize=2,
            label=fr"$h={h:g}$",
        )

        ax_c.errorbar(
            beta,
            data["c_h"],
            yerr=data["c_h_err"],
            marker="o",
            lw=1.5,
            ms=4,
            capsize=2,
            label=fr"$h={h:g}$",
        )

        ax_om.errorbar(
            beta,
            data["omega_beta_h"],
            yerr=data["omega_err"],
            marker="o",
            lw=1.5,
            ms=4,
            capsize=2,
            label=fr"$h={h:g}$",
        )

    beta_c = 0.5 * np.log(1 + np.sqrt(2))

    for ax in axes.ravel():
        ax.axvline(beta_c, color="k", ls="--", lw=1, alpha=0.7)
        ax.set_xlabel(r"$\beta$")
        ax.grid(alpha=0.25)

    ax_m.set_ylabel(r"$\langle m\rangle$")
    ax_chi.set_ylabel(r"$\chi = \beta N\,\mathrm{Var}(m)$")
    ax_c.set_ylabel(r"$C_h = \beta^2 N\,\mathrm{Var}(e)$")
    ax_om.set_ylabel(r"$\Omega_{\beta h}=-N\,\mathrm{Cov}(e,m)$")

    ax_m.set_title("Magnetization")
    ax_chi.set_title("Magnetic susceptibility")
    ax_c.set_title("Specific heat")
    ax_om.set_title("Mixed curvature response")

    ax_chi.legend(frameon=True, fontsize=9)
    fig.tight_layout()
    plt.show()


if __name__ == "__main__":
    L = 64
    J = 1.0

    beta_c = 0.5 * np.log(1 + np.sqrt(2))

    # Representative fields for reviewer-friendly cross sections.
    fields = [0.02, 0.05, 0.10, 0.20]

    # Denser near beta_c and ridge region.
    betas = np.concatenate([
        np.linspace(0.28, 0.36, 9),
        np.linspace(0.365, 0.46, 20),
        np.linspace(0.465, 0.55, 10),
    ])

    betas = np.unique(np.round(betas, 5))

    all_data = {}

    for h in fields:
        print(f"\n=== Scanning h = {h} ===")
        results = scan_beta_for_field(
            L=L,
            betas=betas,
            h=h,
            J=J,
            n_therm=5000,
            n_sweeps=30000,
            sample_every=10,
            seed=2026,
        )

        data = results_to_arrays(results)
        all_data[h] = data

        np.savetxt(
            f"ising_cross_sections_L{L}_h{h:.3f}.dat",
            np.column_stack([
                data["beta"],
                data["m_mean"],
                data["m_err"],
                data["chi"],
                data["chi_err"],
                data["c_h"],
                data["c_h_err"],
                data["omega_beta_h"],
                data["omega_err"],
            ]),
            header="beta m m_err chi chi_err C_h C_h_err Omega_beta_h Omega_err",
        )

    plot_cross_sections(all_data)


=== Scanning h = 0.02 ===
[2026-06-17 23:05:46]h=0.020, beta=0.2800, m=+0.0315, chi=1.654, C=0.224, Omega=0.416 (elapsed=33.1min)
[2026-06-17 23:05:46]h=0.020, beta=0.2900, m=+0.0357, chi=1.812, C=0.257, Omega=0.521 (elapsed=66.7min)
[2026-06-17 23:05:46]h=0.020, beta=0.3000, m=+0.0432, chi=2.175, C=0.292, Omega=0.711 (elapsed=99.7min)
[2026-06-17 23:05:46]h=0.020, beta=0.3100, m=+0.0502, chi=2.515, C=0.339, Omega=0.723 (elapsed=132.4min)
[2026-06-17 23:05:46]h=0.020, beta=0.3200, m=+0.0602, chi=2.957, C=0.364, Omega=1.011 (elapsed=164.9min)
[2026-06-17 23:05:46]h=0.020, beta=0.3300, m=+0.0724, chi=3.599, C=0.427, Omega=1.431 (elapsed=197.5min)
